# Fixing a state

A chosen scissor mechanism is constructed, its length and hinge loations changed, 

Additional Dependency: Pygmo is currently not included in the quivira `environment.yml`.

In [1]:
import numpy as np
import heyoka as hy
import quivira as qv
from _animation import animate_bar_model
# Time measurement
import time

## 1. Build a scissor mechanism and a valid initial state

A 4-bar scissor mechansim is constructed. The bars are connected by joints. Two supports constrain the motion of the mechanism.

Joints are defined by the indices of the bars they join together and the respective (normalised) position along the bars.

For a bar with midpoint position $(x, y)$ and angle $\theta$, we describe a point on the bar by
$$
\mathbf{p}(s) = \begin{bmatrix} x \\ y \end{bmatrix}
+ s\,L \begin{bmatrix} \cos\theta \\ \sin\theta \end{bmatrix},
$$
where $L$ is the bar length and $s \in [-\tfrac{1}{2}, \tfrac{1}{2}]$ is the normalized position along the bar measured from its center. The endpoints correspond to $s = -\tfrac{1}{2}$ and $s = \tfrac{1}{2}$, while intermediate locations use any value in that interval.

We use the following parameters:

- par[0] ... par[3]: The length of each bar
- par[4], par[5]: The values $s$ of the first two bars that define the joint location
- par[6]: The force value driving the one degree of freedom of the mechanism

The ODE system is built from this information only. Its initial condition must be provided separately and
be compatible with the constraints imposed by the joints and supports.


In [2]:
# Instantiate an empty scissor mechanism model
scissor = qv.bar_joint_model(tolerance=1e-8)

# Define bar lengths and joint as heyoka parameters
l0 = hy.par[0]
l1 = hy.par[1]
l2 = hy.par[2]
l3 = hy.par[3]
s_joint_bars_0_1 = hy.par[4]
s_joint_bars_1_0 = hy.par[5]

# Define material density per unit length
rho = 1

# Define the topology
# Use add_bar() to define the bars explicitly
scissor.add_bar(length=l0, mass=l0*rho)
scissor.add_bar(length=l1, mass=l1*rho)
scissor.add_bar(length=l2, mass=l2*rho)
scissor.add_bar(length=l3, mass=l3*rho)

# Use add_joint() to define the joints explicitly
scissor.add_joint(bar_index=[0, 1], s=[s_joint_bars_0_1, s_joint_bars_1_0])
scissor.add_joint(bar_index=[0, 3], s=[0.5, -0.5])
scissor.add_joint(bar_index=[1, 2], s=[0.5, -0.5])
scissor.add_joint(bar_index=[2, 3], s=[0.5, 0.5])

# Add supports
scissor.add_support(bar_index=0, s=-0.5, dof_in_x=False, dof_in_y=False, x=0.0, y=0.0)
scissor.add_support(bar_index=1, s=-0.5, dof_in_x=False, dof_in_y=True, x=0.0, y=4.0)

# Add an external force at the beginning of bar 1
Fy = hy.par[6]
scissor.add_external_force(bar_index=1, s=-0.5, fx=0.0, fy=Fy)

We can now build the ODE system for the scissor mechanism and define its initial state.

In [ ]:
# Build the ODE
ode_system = scissor.build_lagrange_eom(verbose=False)

# Define initial state
scissor.set_bar_pose(bar_index=0, x0=2, y0=2, theta0=np.deg2rad(45))
scissor.set_bar_pose(bar_index=1, x0=2, y0=2, theta0=np.deg2rad(-45))
scissor.set_bar_pose(bar_index=2, x0=5, y0=1, theta0=np.deg2rad(45))
scissor.set_bar_pose(bar_index=3, x0=5, y0=3, theta0=np.deg2rad(-45))

# Bar lengths:
param_values0 = [4 * np.sqrt(2), 4 * np.sqrt(2), 2 * np.sqrt(2), 2 * np.sqrt(2), 0.0, 0.0]

#Check the initial state
state0 = np.asarray(scissor.get_state(), dtype=float)
valid = scissor.check_state(state0, param_values0, verbose=True)

# Set the force value:
param_values0.append(50.0)

Position constraint 0: fulfilled (residual = 0.0)
Position constraint 1: fulfilled (residual = 0.0)
Position constraint 2: fulfilled (residual = 0.0)
Position constraint 3: fulfilled (residual = 0.0)
Position constraint 4: fulfilled (residual = 0.0)
Position constraint 5: fulfilled (residual = -2.0293072715139053e-17)
Position constraint 6: fulfilled (residual = 0.0)
Position constraint 7: fulfilled (residual = 0.0)
Position constraint 8: fulfilled (residual = -2.7343234630647693e-16)
Position constraint 9: fulfilled (residual = 4.0586145430278105e-17)
Position constraint 10: fulfilled (residual = -2.7343234630647693e-16)
Velocity constraint 0: fulfilled (residual = 0.0)
Velocity constraint 1: fulfilled (residual = 0.0)
Velocity constraint 2: fulfilled (residual = 0.0)
Velocity constraint 3: fulfilled (residual = 0.0)
Velocity constraint 4: fulfilled (residual = 0.0)
Velocity constraint 5: fulfilled (residual = 0.0)
Velocity constraint 6: fulfilled (residual = 0.0)
Velocity constraint 

Basic collision events are defined to stop simulations early, to avoid singularities.

In [4]:
# Terminal events: prevent joint/support coincidences
d_min = 0.5

# Symbolic positions of all joints, using their first connected bar.
# This automatically accommodates parametrized lengths and joint s values.
joint_positions = [
    scissor._expr_point_on_bar(j.bar_index[0], j.s[0])
    for j in scissor.joints
]

# Symbolic positions of the actual support points.
# Important: the slider's y coordinate is free to move.
support_positions = [
    scissor._expr_point_on_bar(s.bar_index, s.s)
    for s in scissor.supports
]

# Pairs to monitor
collision_pairs = [
    ("Joint 0 - Joint 3", joint_positions[0], joint_positions[3]),
    ("Joint 1 - Joint 2", joint_positions[1], joint_positions[2]),
    ("Joint 1 - Support 0", joint_positions[1], support_positions[0]),
    ("Joint 1 - Support 1", joint_positions[1], support_positions[1]),
    ("Joint 2 - Support 0", joint_positions[2], support_positions[0]),
    ("Joint 2 - Support 1", joint_positions[2], support_positions[1]),
]

collision_events = []

for name, (x1, y1), (x2, y2) in collision_pairs:
    if name == "Joint 2 - Support 0":
        # Only monitor absolute x separation
        distance_sq = (x1 - x2)**2
    else:
        # Monitor Euclidean distance
        distance_sq = (x1 - x2)**2 + (y1 - y2)**2

    collision_events.append(hy.t_event(distance_sq - d_min**2, direction=hy.event_direction.negative))

The taylor adaptive integrator can now be built.

With a positive force value in y, the state is advanced using the integrator to a "stowed" configuration called `state1`, stopped by the events defined above.

Reversing the force, setting the velocities of `state1` to 0, and integrating again leads to a deployed configuration `state2`.

The transition `state1` -> `state2` is animated.

In [5]:
ta = hy.taylor_adaptive(
    ode_system,
    pars=param_values0,
    t_events=collision_events,
    compact_mode=True
)

ta.state[:] = state0
t_end = 20.0
tgrid = np.linspace(0.0, t_end, 401)
result1 = ta.propagate_grid(tgrid)

# Integrate until the first collision threshold is reached
solution1 = np.asarray(result1[-1], dtype=float)

# Save the exact final state and time of the first integration
state1 = ta.state.copy()
t1 = ta.time

# Stop the mechanism at the turning point
# This sets all generalized velocities to zero.
nq = len(scissor.q)
ta.state[nq:] = 0.0

# The new parameters are the old one but the force is reversed
param_values1 = np.asarray(param_values0, dtype=float).copy()
param_values1[6] = -50.0
ta.pars[:] = param_values1

# Integrate for another 20 seconds, starting at the current time
tgrid2 = t1 + np.linspace(0.0, t_end, 401)

result2 = ta.propagate_grid(tgrid2)
solution2 = np.asarray(result2[-1], dtype=float)

print("Final time:", ta.time)
print("Number of frames:", len(solution2))

animate_bar_model(
    scissor,
    solution2,
    pars=ta.pars,
    save_animation=False
)

Final time: 5.64020554684806
Number of frames: 80


## 2. Adjust the parameters

We adjust the coordinates s where the first hinge attaches to each of the bars, and the length of the other two bars.

In [5]:
# Adjust heyoka parameters
param_values3 = np.asarray(ta.pars, dtype=float).copy()
param_values3[4] = +0.1
param_values3[5] = -0.1
param_values3[2] = param_values3[0]/2-0.1*param_values3[0]
param_values3[3] = param_values3[1]/2+0.1*param_values3[1]

## 3. Formulation as a constrained optimization problem

After changing the bar lengths and joint attachment positions, the previous configuration generally no longer satisfy the geometric constraints of the mechanism. We therefore need to find a new valid configuration that remains as close as possible to the original one.

We formulate this as a constrained optimization problem:

$$
\boxed{
\begin{aligned}
\min_{\mathbf q}\quad & D(\mathbf q,\mathbf q_{\mathrm{ref}})\
\text{subject to}\quad & \mathbf F(\mathbf q,\mathbf p_{\mathrm{new}})=0
\end{aligned}
}
$$

Here:

$\mathbf q_{\mathrm{ref}}$ is the configuration before reconfiguration.

$\mathbf q$ contains the new bar positions and orientations.

$\mathbf p_{\mathrm{new}}$ contains the updated geometric parameters, including bar lengths and joint attachment coordinates.

$\mathbf F=0$ enforces the geometric constraints of the mechanism.

$D$ measures how much the configuration changes.

The objective is to minimize the movement of the mechanism while satisfying all geometric constraints.

Furthermore, we define the Jacobian $\mathbf J=\partial\mathbf F/\partial\mathbf q$, which will tell the optimizer how the constraints change when the bar coordinates change.

Since generalized coordinates contain both translations (meters) and rotations (radians), a simple Euclidean distance between coordinate vectors is not physically meaningful. We therefore consider two geometrically motivated distance measures based on the displacement of bar endpoints: a quadratic approximation and an exact endpoint-displacement metric.

## 4. Defining the optimization metrics from the movement of the endpoints

The endpoint positions are

$$
\boxed{
\mathbf r^\pm(\mathbf q,L)
=
\begin{bmatrix}x\\y\end{bmatrix}
\pm\frac L2
\begin{bmatrix}\cos\theta\\\sin\theta\end{bmatrix}
}
$$

For convenience, define the orientation vector

$$
\mathbf u(\theta)=
\begin{bmatrix}\cos\theta\\\sin\theta\end{bmatrix}.
$$

### 4.1 Using the exact endpoints

We obtain the **exact endpoint metric**:

$$
\boxed{
D_{\mathrm{end}}=
\Delta x^2+\Delta y^2+
\frac{L_1^2+L_0^2-2L_1L_0\cos\Delta\theta}{4}.
}
$$

This expression measures the actual squared distance between the old and new endpoint positions, even when the bar rotates substantially or changes length. Its derivative is:

$$
\boxed{
\nabla_{\mathbf q}D_{\mathrm{end}}
=
\begin{bmatrix}
2\Delta x\\
2\Delta y\\
\frac{L_1L_0}{2}\sin\Delta\theta
\end{bmatrix}.
}
$$

In [ ]:
state3 = scissor.fix_state(state=state1, pars=param_values3, set_velocities_zero=True, use_exact_endpoints=True)

ta.state[:] = state3
ta.pars[:] = param_values3
ta.time = t1

# Integrate for another 20 seconds, starting at the current time
tgrid3 = t1 + np.linspace(0.0, t_end, 401)

result3 = ta.propagate_grid(tgrid3)
solution3 = np.asarray(result3[-1], dtype=float)

print("Final time:", ta.time)
print("Number of frames:", len(solution3))

animate_bar_model(
    scissor,
    solution3,
    pars=ta.pars,
    save_animation=False
)

Final time: 6.375130285955375
Number of frames: 95


### 4.2 Approximate Endpoint Metric

Using a small angle approximation, we can obtain a simplified **quadratic form**:

$$
\boxed{
D_R = \frac12 \Delta\mathbf q^T \mathbf R \Delta\mathbf q,
}
$$

where

$$
\Delta\mathbf q =
\mathbf q-\mathbf q_{\mathrm{ref}}
=
\begin{bmatrix}
\Delta x\\
\Delta y\\
\Delta\theta
\end{bmatrix}.
$$

and 

$$
\mathbf R=\operatorname{diag}\left(
2,\;2,\;\frac{L_1L_0}{2}
\right).
$$

The factors of 2 compensate for the prefactor $1/2$ in the quadratic form.

**Why use this representation?** Since $\mathbf R$ is symmetric and constant during optimization, the objective gradient becomes particularly simple:

$$
\boxed{
\nabla_{\mathbf q}D_R
=
\mathbf R(\mathbf q-\mathbf q_{\mathrm{ref}}).
}
$$

For the complete mechanism, we construct a block-diagonal matrix containing one such $\mathbf R_i$ for each bar. Because all blocks are diagonal, we only need to store the diagonal entries as `R_diag`.

This gives us a computationally inexpensive objective and analytical gradient for the optimization. Unlike the exact endpoint metric, the quadratic approximation does not require evaluating trigonometric functions during optimization.

In [ ]:
state4 = scissor.fix_state(state=state1, pars=param_values3, set_velocities_zero=True, use_exact_endpoints=False)

ta.state[:] = state4
ta.pars[:] = param_values3
ta.time = t1

# Integrate for another 20 seconds, starting at the current time
tgrid4 = t1 + np.linspace(0.0, t_end, 401)

result4 = ta.propagate_grid(tgrid4)
solution4 = np.asarray(result4[-1], dtype=float)

print("Final time:", ta.time)
print("Number of frames:", len(solution4))

animate_bar_model(
    scissor,
    solution4,
    pars=ta.pars,
    save_animation=False
)

Final time: 6.375130214245181
Number of frames: 95


## 5. Find a valid state from a random initial guess

This could be useful if only some or none of the starting conditions are known.

In [ ]:
rng = np.random.default_rng(42)

n_bars = len(scissor.bars)
L_scale = max(param_values3[:4])

# Generate completely random bar positions and orientations
q_random = np.empty(nq, dtype=float)
q_random[0::3] = rng.uniform(-2.0 * L_scale, 2.0 * L_scale, size=n_bars)
q_random[1::3] = rng.uniform(-2.0 * L_scale, 2.0 * L_scale, size=n_bars)
q_random[2::3] = rng.uniform(-np.pi, np.pi, size=n_bars)

# fix_state expects a full state: (q, qdot)
state_random = np.concatenate((q_random, np.zeros(nq)))

# Project the random initial condition onto the constrained manifold
state5 = scissor.fix_state(
    state=state_random,
    pars=param_values3,
    set_velocities_zero=True,
    use_exact_endpoints=False,
)

ta.state[:] = state5
ta.pars[:] = param_values3
ta.time = t1

# Integrate from the corrected random initial condition
tgrid5 = t1 + np.linspace(0.0, t_end, 401)
result5 = ta.propagate_grid(tgrid5)
solution5 = np.asarray(result5[-1], dtype=float)

print("Final time:", ta.time)
print("Number of frames:", len(solution5))

animate_bar_model(
    scissor,
    solution5,
    pars=ta.pars,
    save_animation=False
)

Final time: 3.243079326489089
Number of frames: 32
